# 05 Decision Tree With Basic Image Features

This notebook compares an untuned `DecisionTreeClassifier` with a simply tuned tree using the existing basic image features. The goal is to reduce overfitting, not to maximize raw accuracy.

## 1. Project setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

## 2. Imports

In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mse446_matplotlib")

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from src.extract_basic_features import BASIC_FEATURE_COLUMNS
from src.train_decision_tree import (
    CLASS_LABELS,
    CONFUSION_MATRIX_FIGURE,
    FEATURES_CSV,
    SCORES_CSV,
    TUNED_CONFUSION_MATRIX_FIGURE,
    TUNED_SCORES_CSV,
    evaluate_model,
    make_model,
    print_overfitting_summary,
    tune_decision_tree,
)
from src.train_logistic_regression import choose_group_split, load_features

## 3. Paths and configuration

In [ ]:
print(f"Basic features CSV: {FEATURES_CSV}")
print(f"Untuned scores CSV: {SCORES_CSV}")
print(f"Tuned scores CSV: {TUNED_SCORES_CSV}")

## 4. Load metadata

In [ ]:
features = load_features(FEATURES_CSV)
features.head()

## 5. Sanity checks

In [ ]:
print(f"Rows: {len(features)}")
print("Label counts:")
print(features["label"].value_counts().sort_index())
print(f"Image feature columns: {len(BASIC_FEATURE_COLUMNS)}")
assert set(BASIC_FEATURE_COLUMNS).issubset(features.columns)
assert features["area_group"].notna().all()

## 6. Main analysis

The held-out split is selected using the same grouped split helper as Logistic Regression. Hyperparameter tuning uses `GroupKFold` only inside the training split, scored with macro F1.

In [ ]:
train_idx, test_idx, selected_seed, split_distance = choose_group_split(features)

groups = features["area_group"]
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))

X = features[BASIC_FEATURE_COLUMNS]
y = features["label"]
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]
train_groups = groups.iloc[train_idx]

basic_scores, basic_report, basic_matrix, basic_figure = evaluate_model(
    make_model(),
    "decision_tree_basic",
    X_train,
    X_test,
    y_train,
    y_test,
    selected_seed,
    split_distance,
    CONFUSION_MATRIX_FIGURE,
    "Decision tree basic",
)

search = tune_decision_tree(X_train, y_train, train_groups)
tuned_scores, tuned_report, tuned_matrix, tuned_figure = evaluate_model(
    search.best_estimator_,
    "decision_tree_tuned",
    X_train,
    X_test,
    y_train,
    y_test,
    selected_seed,
    split_distance,
    TUNED_CONFUSION_MATRIX_FIGURE,
    "Decision tree tuned",
    float(search.best_score_),
    search.best_params_,
)

tuned_scores

## 7. Results/figures

In [ ]:
basic_scores.to_csv(SCORES_CSV, index=False)
tuned_scores.to_csv(TUNED_SCORES_CSV, index=False)

print_overfitting_summary(basic_scores, "Untuned decision tree")
print_overfitting_summary(tuned_scores, "Tuned decision tree")
print("Best CV macro F1:", round(search.best_score_, 4))
print("Best params:", search.best_params_)
print("\nUntuned report:")
print(basic_report)
print("Tuned report:")
print(tuned_report)
print(f"Saved untuned scores to {SCORES_CSV}")
print(f"Saved tuned scores to {TUNED_SCORES_CSV}")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    search.best_estimator_.predict(X_test),
    labels=CLASS_LABELS,
    cmap="Blues",
    colorbar=False,
)
plt.title("Decision tree tuned")
plt.show()

## 8. Notes for report

- The untuned tree is expected to overfit if train metrics are much higher than test metrics.
- The tuned tree restricts complexity using depth and sample-count controls.
- This step still uses only basic image features and does not add RandomForest, SVC, KNN, HOG, graph features, or CNNs.